# 3. Baseline Credit Risk Model

This notebook develops a baseline machine learning model for credit
default prediction.

The main objectives are:

- Load the prepared training and testing data
- Define numerical and categorical features
- Build a preprocessing pipeline
- Train a Logistic Regression baseline model
- Generate default probabilities
- Evaluate classification performance
- Evaluate probability quality using appropriate metrics

The baseline model will provide a reference point for later model
comparison, probability calibration, risk scoring, and financial loss
analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    brier_score_loss,
    log_loss
)

In [2]:
# Load the processed datasets

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (24000, 39)
X_test: (6000, 39)
y_train: (24000,)
y_test: (6000,)


In [3]:
# Identify categorical and numerical features

categorical_features = [
    "SEX",
    "EDUCATION",
    "MARRIAGE"
]

numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

print("Categorical features:")
print(categorical_features)

print("\nNumber of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))
print("Total features:", len(numerical_features) + len(categorical_features))

Categorical features:
['SEX', 'EDUCATION', 'MARRIAGE']

Number of numerical features: 36
Number of categorical features: 3
Total features: 39


Build the preprocessing pipeline

In [4]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [5]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

### Create the Logistic Regression pipeline

In [6]:
baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight=None,
        random_state=42
    ))
])

In [7]:
# Train the model

baseline_model.fit(X_train, y_train)

print("Baseline Logistic Regression model trained successfully.")

Baseline Logistic Regression model trained successfully.


Make predictions

In [8]:
# predicted class 
y_pred = baseline_model.predict(X_test)

# predicted probability
y_proba = baseline_model.predict_proba(X_test)[:, 1]

Evaluate the baseline

In [9]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

brier = brier_score_loss(y_test, y_proba)
logloss = log_loss(y_test, y_proba)

print("Baseline Model Performance")
print("=" * 40)

print(f"Accuracy:           {accuracy:.4f}")
print(f"Balanced Accuracy:  {balanced_acc:.4f}")
print(f"Precision:          {precision:.4f}")
print(f"Recall:             {recall:.4f}")
print(f"F1 Score:           {f1:.4f}")
print(f"ROC-AUC:            {roc_auc:.4f}")
print(f"PR-AUC:             {pr_auc:.4f}")
print(f"Brier Score:        {brier:.4f}")
print(f"Log Loss:           {logloss:.4f}")

Baseline Model Performance
Accuracy:           0.8082
Balanced Accuracy:  0.6254
Precision:          0.6433
Recall:             0.2977
F1 Score:           0.4070
ROC-AUC:            0.7432
PR-AUC:             0.5031
Brier Score:        0.1428
Log Loss:           0.4522


Confusion matrix

In [10]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print(classification_report(
    y_test,
    y_pred,
    target_names=["Non-Default", "Default"]
))

Confusion Matrix:
[[4454  219]
 [ 932  395]]
              precision    recall  f1-score   support

 Non-Default       0.83      0.95      0.89      4673
     Default       0.64      0.30      0.41      1327

    accuracy                           0.81      6000
   macro avg       0.74      0.63      0.65      6000
weighted avg       0.79      0.81      0.78      6000



Probability Check

In [11]:
print("Probability statistics:")
print(pd.Series(y_proba).describe())

print("\nFirst 10 predicted default probabilities:")
print(y_proba[:10])

Probability statistics:
count    6.000000e+03
mean     2.202679e-01
std      1.812185e-01
min      3.947050e-07
25%      1.065283e-01
50%      1.407782e-01
75%      2.728686e-01
max      9.151412e-01
dtype: float64

First 10 predicted default probabilities:
[0.13628701 0.16158187 0.15723757 0.10548479 0.05980636 0.43694126
 0.11189928 0.07954045 0.07401702 0.11230291]
